In [96]:
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

torch.manual_seed(40)

In [97]:
df = pd.read_csv('../3. Dataset/Dataset_English_Hindi.csv')
df.tail(3)

,English,Hindi
130473,"As for the other derivatives of sulphur , the ...","जहां तक गंधक के अन्य उत्पादों का प्रश्न है , द..."
130474,its complicated functioning is defined thus in...,Zरचना-प्रकिया को उसने एक पहेली में यों बांधा है .
130475,They've just won four government contracts to ...,हाल ही में उन्हें सरकारी ठेका मिला है करीब सौ ...


In [98]:
df.isna().sum()

English      2
Hindi      312
dtype: int64

In [99]:
df = df.dropna(subset=['English', 'Hindi']).reset_index(drop=True)

In [100]:
df.isna().sum()

English    0
Hindi      0
dtype: int64

In [101]:
def tokenize(text):
  
  text = text.lower()
  text = text.replace('?','')
  text = text.replace("'","")
  
  return text.split()


def build_vocab(sentences):
  
  vocab = {'<UNK>':1 , '<PAD>' : 0}

  for sentence in sentences :
    tokenized_sentence = tokenize(sentence)
    for token in tokenized_sentence:
       if token not in vocab:     
            vocab[token] = len(vocab)
        
  return vocab

vocab_en = build_vocab(df['English'].values)
vocab_hi = build_vocab(df['Hindi'])

print(len(vocab_en) , len(vocab_hi))

100963 94945


In [102]:
def text_to_indices(text, vocab):
  
  indexed_text = []
  for token in tokenize(text):

    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])

  return indexed_text

In [103]:
train_df , test_df = train_test_split(df , test_size=0.2  , random_state=20)

In [104]:
class TranslationDataset(Dataset):

    def __init__(self, df,src_vocab,tgt_vocab):
        
        self.df = df
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab

    def __len__(self):

        return self.df.shape[0]
    
    def __getitem__(self, idx):

        english_sentence = text_to_indices(self.df.iloc[idx]['English'], self.src_vocab)
        hindi_sentence = text_to_indices(self.df.iloc[idx]['Hindi'], self.tgt_vocab)

        return torch.tensor(english_sentence), torch.tensor(hindi_sentence)


In [105]:
def collate_fn(batch):

    src_batch, tgt_batch = zip(*batch)
    
    src_batch = pad_sequence(src_batch, batch_first=True, padding_value=vocab_en['<PAD>'])
    tgt_batch = pad_sequence(tgt_batch, batch_first=True, padding_value=vocab_hi['<PAD>'])
    
    # note : it consider max_len for each batch.

    return src_batch, tgt_batch


In [106]:
train_dataset = TranslationDataset(df,vocab_en,vocab_hi)
train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)

In [107]:
src_vocab_size = len(vocab_en)
tgt_vocab_size = len(vocab_hi)

src_seq_len = 512  # used to caculate positinal embeddings
tgt_seq_len = 512  # used to caculate positinal embeddings

d_model = 512 # embedding dim
N = 6 # no of layers 
h = 8 # no of heads (d_model % h == 0)
dropout = 0.1 
d_ff = 2048 # neurons in hidden layer


In [108]:
from models.Transformer_Architecture import build_transformer

model = build_transformer(src_vocab_size, tgt_vocab_size , src_seq_len , tgt_seq_len , d_model , N ,h , dropout , d_ff ) 

In [109]:
# check this how it works

def create_src_mask(src, pad_idx=0):
    # src: (batch, src_len)
    return (src != pad_idx).unsqueeze(1).unsqueeze(2)
    # (batch, 1, 1, src_len)

def subsequent_mask(size):
    return torch.tril(torch.ones(size, size)).bool()

def create_tgt_mask(tgt, pad_idx=0):

    batch, tgt_len = tgt.shape
    pad_mask = (tgt != pad_idx).unsqueeze(1).unsqueeze(2)
    causal_mask = subsequent_mask(tgt_len).to(tgt.device)
    return pad_mask & causal_mask


In [110]:
batch = next(iter(train_dataloader))
src, tgt = batch

src_mask = create_src_mask(src)
tgt_mask = create_tgt_mask(tgt)


print(src.shape , tgt.shape , src_mask.shape , tgt_mask.shape)


torch.Size([4, 30]) torch.Size([4, 30]) torch.Size([4, 1, 1, 30]) torch.Size([4, 1, 30, 30])


In [111]:
from torchinfo import summary

summary(
    model,
    input_data=(src,tgt,src_mask,tgt_mask)
)

Layer (type:depth-idx)                                  Output Shape              Param #
Transformer                                             [4, 30, 94945]            --
├─InputEmbeddings: 1-1                                  [4, 30, 512]              --
│    └─Embedding: 2-1                                   [4, 30, 512]              51,693,056
├─PositionalEncoding: 1-2                               [4, 30, 512]              --
│    └─Dropout: 2-2                                     [4, 30, 512]              --
├─Encoder: 1-3                                          [4, 30, 512]              --
│    └─ModuleList: 2-3                                  --                        --
│    │    └─EncoderBlock: 3-1                           [4, 30, 512]              3,150,336
│    │    └─EncoderBlock: 3-2                           [4, 30, 512]              3,150,336
│    │    └─EncoderBlock: 3-3                           [4, 30, 512]              3,150,336
│    │    └─EncoderBlock: 3-4  

In [112]:
learning_rate = 0.001
epochs = 5

criterion = torch.nn.CrossEntropyLoss(ignore_index=vocab_en['<PAD>'])
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Training

In [113]:
from tqdm import tqdm

for epoch in range(epochs):
    
    total_loss = 0

    # tqdm wraps the dataloader
    for batch in tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{epochs}", leave=False):
        
        src, tgt = batch

        src_mask = create_src_mask(src)
        tgt_mask = create_tgt_mask(tgt)

        optimizer.zero_grad()

        output = model(src, tgt, src_mask, tgt_mask)

        b, t, v = output.shape
        loss = criterion(output.reshape(b*t, v), tgt.reshape(b*t))
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        break

    print(f"Epoch: {epoch+1}, Loss: {total_loss:.4f}")


Epoch: 1, Loss: 11.4751


Epoch: 2, Loss: 11.4466


Epoch: 3, Loss: 11.3381


Epoch: 4, Loss: 10.9869


Epoch: 5, Loss: 10.8730


# Testing 

# Prediction for single 

# Training on GPU

In [114]:
device = 'cpu'
if hasattr(torch,'mps') and torch.backends.mps.is_available():
    device = 'mps'
    print("MPS is available")

MPS is available


In [ ]:
model.to(device)

for epoch in range(epochs):
    
    total_loss = 0

    # tqdm wraps the dataloader
    for batch in tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{epochs}", leave=False):
        
        src, tgt = batch
        src = src.to(device)
        tgt = tgt.to(device)

        src_mask = create_src_mask(src).to(device)
        tgt_mask = create_tgt_mask(tgt).to(device)

        optimizer.zero_grad()

        output = model(src, tgt, src_mask, tgt_mask)

        b, t, v = output.shape
        loss = criterion(output.reshape(b*t, v), tgt.reshape(b*t))
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        break

    print(f"Epoch: {epoch+1}, Loss: {total_loss:.4f}")
